# $B^+ \to \pi^+\pi^+\pi^-$ — Jax-PWA gen-fit study without CP violation: **isobar generation, QMIPixel fit** (LHCb Run 1 model)

Simplified, CP-conserving version of `b2pipipi_cpvfit_isobar_gen_qmi_pixel_fit_toy_pull_study_SqDP.ipynb`:
only $B^+$ events, the CP-averaged isobar coefficients $(x, y)$ of Table XXI, and one magnitude and
one phase per QMIPixel cell. The toys are generated with the **isobaric LHCb Run 1 model** (S-wave =
`sigma` + `rescattering`) and fitted with the same P/D/F waves plus a **`QMIPixel`** S-wave. The
questions are

1. whether the P/D/F-wave parameters stay unbiased when the S-wave is replaced by the QMIPixel, and
2. whether the QMIPixel recovers, cell by cell, the magnitude and phase of the isobaric S-wave
   $S_\text{gen}(s_{12}, s_{13}) = S_\text{iso}(s_{12}) + S_\text{iso}(s_{13})$.

On the folded plane ($s_\text{low} = \min(s_{12}, s_{13})$, $s_\text{high} = \max(s_{12}, s_{13})$)
the isobaric S-wave depends mostly on $s_\text{low}$.

**Expected limitation.** A free 2D field has no spin-0 angular structure: at fixed $s_\text{low}$
the angular dependence of a P or D wave is a smooth function of $s_\text{high}$ that the
bilinear cell interpolation can reproduce. The P/D/F coefficients are therefore correlated with
the cells; the fraction of each P/D wave that the grid cannot reproduce is printed below
("Generated S-wave and the uniform grid").

## Model

P/D/F waves: the LHCb Run 1 isobar model of
[Phys. Rev. D 101, 012006 (2020)](https://arxiv.org/abs/1909.05212):

| component | lineshape | spin | mass, width [GeV] |
|---|---|---|---|
| `rho770` | Gounaris-Sakurai | 1 | 0.7708, 0.1534 (Table XX) |
| `omega782` | relativistic Breit-Wigner | 1 | 0.78265, 0.00849 |
| `f2_1270` | relativistic Breit-Wigner | 2 | 1.256, 0.1867 |
| `rho1450` | relativistic Breit-Wigner | 1 | 1.465, 0.400 |
| `rho3_1690` | relativistic Breit-Wigner | 3 | 1.686, 0.186 |
| `sigma` (generation) | `SigmaPole` | 0 | 0.563, 0.350 (Table XXII) |
| `rescattering` (generation) | `PipiKKRescattering` (Laura++) | 0 | — |
| `S_QMIPixel` (fit) | `DalitzAmplitude`, unfolded `QMIPixel`, linear, mirrored cells | 0 | adaptive 1D binning on both axes |

**Daughter order.** The QMIPixel is defined on $(s_{12}, s_{13})$ and made symmetric by mirroring
$s_{12} \leftrightarrow s_{13}$, so `s12` and `s13` must be the two $\pi^+\pi^-$ combinations: the final state is ordered $(\pi^-, \pi^+, \pi^+)$ and the
resonances use `pair=(0, 1)` (symmetrized automatically with `(0, 2)`); the Square-Dalitz
normalization uses the $\pi^+\pi^+$ pair `(1, 2)`. It is checked numerically below that this
gives the same intensities as the $(\pi^+, \pi^+, \pi^-)$ ordering of the 1D notebooks.

Coefficients are $c = x + i\,y$ (CP-averaged values of Table XXI), with `rho770.x = 1`,
`rho770.y = 0` fixed as the reference. Components are unit-normalized, except the S-wave
(`normalize_component=False`), so the cells carry the physical S-wave amplitude.

**QMIPixel S-wave.** Each active cell $(i, j)$ has two free `Parameter.dynamics` (owner `S_QMIPixel`):
the magnitude $a_{ij}$ and phase $\varphi_{ij}$ of the S-wave at the cell centre, interpolated
bilinearly between centres. **Reference values** are $S_\text{gen}$ at the cell centres, with
phases made continuous across neighbouring cells. As in the 1D study they are reference, not
true, values: any residual bias measures the grid approximation, not a fitter bias.

## Generation and fit

- **Pure signal**, $B^+$ only, generator level: no efficiency, no background, no veto.
- Toys are generated from the isobar model with `generate_toy(..., method="accept-reject")`.
- Each toy is fitted with the QMIPixel model through `FitSession`, starting from the reference
  values, with HESSE uncertainties.
- Toys are **not** stored. Seeds are deterministic per toy (`SeedSequence([TOY_BASE_SEED, iexpt])`).

## Local caching

`CACHE_DIR` (git-ignored) holds the persistent JAX compilation cache (generator programs only;
per-toy fit programs are discarded) and `RESULTS_CSV`, appended after every toy, so the loop can
be interrupted and resumed.

In [ ]:
import os
from pathlib import Path

os.environ["JAX_PLATFORMS"] = "cuda"  # cpu, cuda, tpu
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Local cache root; set before JAX is imported so the persistent compilation cache is active
# from the first compiled program.
CACHE_DIR = Path(".cache/b2pipipi_lhcb_run1_isobar_gen_qmi_pixel_fit_no_cp").resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(CACHE_DIR / "jax_compilation"))
os.environ.setdefault("JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS", "1.0")

In [ ]:
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "jaxpwa").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break

import time
from collections import deque

import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np
import pandas as pd

from jaxpwa import (
    DalitzAmplitude,
    DecayChannel,
    DecayModel,
    FitSession,
    GounarisSakurai,
    Parameter,
    PipiKKRescattering,
    QMIPixel,
    RealImag,
    RelativisticBreitWigner,
    Resonance,
    SigmaPole,
    ZemachP,
    enable_x64,
    generate_toy,
    physical_bin_mask,
)

enable_x64()

## Configuration

In [ ]:
N_TOY_EVENTS = 50_000  # B+ events per toy
N_EXPERIMENTS = 500
TOY_INDICES = range(N_EXPERIMENTS)  # narrow this (e.g. range(3)) for a quick smoke test
TOY_BASE_SEED = 1909052140  # toy iexpt uses SeedSequence([TOY_BASE_SEED, iexpt])
TOY_POOL_SIZE = 1_000_000  # accept-reject pilot; large enough to resolve omega(782)

NORMALIZATION_RESOLUTION = 1000  # Square-Dalitz (0, 1) quadrature, as in the 1D studies
FIT_STRATEGY = 1
FIT_NCALL = 200_000

QMI_PIXEL_INTERPOLATION = "linear"
N_POINTS_1D = 15  # points of the adaptive 1D binning (kinematic limits included)
MIN_POINT_SPACING = 0.05  # [GeV] minimum distance between adaptive points

RESULTS_CSV = CACHE_DIR / (
    f"results_nocp_isogen_qmipixelfit_{QMI_PIXEL_INTERPOLATION}_n{N_TOY_EVENTS}_seed{TOY_BASE_SEED}"
    f"_res{NORMALIZATION_RESOLUTION}_adaptive{N_POINTS_1D}_unfolded.csv"
)
print(f"results: {RESULTS_CSV}")

## Generation model: LHCb isobar model

`ISOBAR_MODELS` is the LHCb isobar model with the numeric CP-averaged Table XXI coefficients $(x, y)$: it is the
**generation model**. Its unit-normalized `sigma` and `rescattering` terms $c_j N_j A_j$ give the
reference S-wave $S_\text{gen}$ evaluated at the QMIPixel cell centres.

In [ ]:
MASS_WIDTHS = {
    "rho770": (0.7708, 0.1534),      # isobar fit, Table XX
    "omega782": (0.78265, 0.00849),  # Laura++/PDG
    "f2_1270": (1.256, 0.1867),      # isobar fit
    "rho1450": (1.465, 0.400),       # Laura++/PDG
    "rho3_1690": (1.686, 0.186),     # Laura++/PDG
    "sigma": (0.563, 0.350),         # fitted pole, Table XXII
}

# name: (x, y, dx, dy), Table XXI; without CP violation only (x, y) are used
COEFFICIENTS = {
    "rho770": (1.000, 0.000, -0.003, 0.000),
    "omega782": (0.091, -0.007, 0.000, -0.022),
    "f2_1270": (0.291, 0.204, -0.002, -0.179),
    "rho1450": (-0.223, 0.191, 0.031, 0.068),
    "rho3_1690": (0.073, -0.045, 0.044, -0.013),
    "rescattering": (0.142, -0.040, -0.047, -0.027),
    "sigma": (-0.485, 0.284, 0.231, 0.270),
}
S_WAVE_ISOBARS = ("rescattering", "sigma")  # replaced by S_QMIPixel in the fit
FIXED_COEFFICIENTS = {"rho770.x", "rho770.y"}  # phase/magnitude reference

LINESHAPES = {
    "rho770": (GounarisSakurai, 1),
    "omega782": (RelativisticBreitWigner, 1),
    "f2_1270": (RelativisticBreitWigner, 2),
    "rho1450": (RelativisticBreitWigner, 1),
    "rho3_1690": (RelativisticBreitWigner, 3),
    "sigma": (SigmaPole, 0),
}


def final_state(charge):
    # Opposite-sign pion first: s12 and s13 are the two pi+ pi- combinations (QMIPixel folding).
    return ("pi-", "pi+", "pi+") if charge > 0 else ("pi+", "pi-", "pi-")


_channel = DecayChannel("B+", final_state(+1))
M_PI = float(_channel.daughter_masses[0])
M_B = float(_channel.parent_mass)
S_MIN = 4 * M_PI**2           # pi+ pi- threshold [GeV^2]
S_MAX = (M_B - M_PI) ** 2     # kinematic maximum of s12 and s13 [GeV^2]
S_SUM = M_B**2 + 3 * M_PI**2  # s12 + s13 + s23
CHARGES = (+1,)  # no CP violation: B+ only


def isobar_resonance(name, coefficient, pair=(0, 1)):
    if name == "rescattering":
        return Resonance(
            name, pair, coefficient,
            lineshape=PipiKKRescattering(convention="laura", delta_kk_squared=2.8**2),
            mass=1.0, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0,
        )
    lineshape, spin = LINESHAPES[name]
    mass, width = MASS_WIDTHS[name]
    return Resonance(
        name, pair, coefficient, lineshape=lineshape(), angular=ZemachP(),
        mass=mass, width=width, spin=spin, resonance_radius=4.0, parent_radius=4.0,
    )


def build_model(charge, components, *, daughters=None, normalization_pair=(1, 2)):
    return DecayModel(
        DecayChannel("B+" if charge > 0 else "B-", daughters or final_state(charge)),
        components,
        normalize_components=True,
        normalization_method="square-dalitz",
        normalization_pair=normalization_pair,
        normalization_resolution=NORMALIZATION_RESOLUTION,
    )


ISOBAR_MODELS = {
    charge: build_model(charge, [
        isobar_resonance(name, RealImag(*values[:2]))
        for name, values in COEFFICIENTS.items()
    ])
    for charge in CHARGES
}


def isobar_s_wave_terms(charge):
    # (c_j^q N_j, A_j) of the unit-normalized sigma and rescattering components.
    model = ISOBAR_MODELS[charge]
    return tuple(
        (complex(np.asarray(component.coefficient.value({})))
         * float(np.asarray(model._component_scale(component, {}))), component.function)
        for component in model.amplitude_model.components
        if component.name in S_WAVE_ISOBARS
    )


def generated_s_wave(charge):
    # Isobaric S-wave S_iso(s12) + S_iso(s13) of the generation model, as a function of data.
    terms = isobar_s_wave_terms(charge)

    def s_wave(data):
        return sum(scale * function(data) for scale, function in terms)

    return s_wave


GEN_MODELS = ISOBAR_MODELS
print("generation components:", [c.name for c in GEN_MODELS[+1].amplitude_model.components])

### Check: the daughter reordering leaves the isobar model unchanged

The LHCb model built with the 1D-notebook ordering $(\pi^\pm, \pi^\pm, \pi^\mp)$, `pair=(2, 0)`,
must give the same intensity as the reordered one at the same physical events.

In [ ]:
_check_sample = ISOBAR_MODELS[+1].generate_phase_space(20_000, seed=1, include_momenta=True)
_new = _check_sample.as_dict()
# old (pi+_a, pi+_b, pi-) <- new (pi-, pi+_a, pi+_b)
_old = {"s12": _new["s23"], "s13": _new["s12"], "s23": _new["s13"],
        "p1": _new["p2"], "p2": _new["p3"], "p3": _new["p1"]}
for charge in CHARGES:
    old_model = build_model(
        charge,
        [isobar_resonance(name, RealImag(*values[:2]), pair=(2, 0))
         for name, values in COEFFICIENTS.items()],
        daughters=final_state(charge)[1:] + final_state(charge)[:1], normalization_pair=(0, 1),
    )
    new_intensity = np.asarray(ISOBAR_MODELS[charge].intensity(_new))
    old_intensity = np.asarray(old_model.intensity(_old))
    deviation = np.max(np.abs(new_intensity / old_intensity - 1.0))
    print(f"charge {charge:+d}: max relative intensity difference = {deviation:.2e}")
    assert deviation < 1e-10

## QMIPixel grid

The pixels are built in two steps.

1. **Adaptive 1D binning, guided by the generated S-wave.** `N_POINTS_1D` points in
   $m(\pi^+\pi^-)$ are chosen one at a time where they most reduce the error of the linear
   interpolation (in $s$, magnitude and phase separately, as `QMIPixel` does) of the isobaric
   S-wave $S_\text{iso}(m)$, weighted by the phase-space density of $m(\pi^+\pi^-)_\text{low}$. The two kinematic limits are always points, and points are kept at
   least `MIN_POINT_SPACING` apart (no close pairs).
2. **2D pixels on the unfolded plane.** The same 1D binning is used on both axes $s_{12}$ and
   $s_{13}$ (the two $\pi^+\pi^-$ combinations). The `QMIPixel` values sit at the cell centres, so
   the bin edges are the midpoints (in $s$) between consecutive points, with the first and last
   bins mirrored around the kinematic limits; the 1D error of step 1 is computed with these actual
   centres. Every grid point $(s_i, s_j)$ that falls **inside the Dalitz plot** gets a free
   magnitude and phase; the other cells are `QMIPixel` ghost cells (nearest active value).

The fit is **unfolded** (`folded=False`), and the identical-pion symmetry is imposed by
**mirrored cells sharing the same parameters**: cell $(i, j)$ and cell $(j, i)$ carry the same
`Parameter` objects, so the field is symmetric under $s_{12} \leftrightarrow s_{13}$ without
doubling the parameters, and the interpolation across the diagonal uses the mirrored value.

In [ ]:
# Phase-space sample (1M uniform events) used for expected yields, grid diagnostics and projections.
_ps = GEN_MODELS[+1].generate_phase_space(1_000_000, seed=7, include_momenta=False)
PS_DATA = {key: jnp.asarray(value) for key, value in _ps.as_dict().items()
           if key in ("s12", "s13", "s23")}
PS_LOW = np.minimum(np.asarray(PS_DATA["s12"]), np.asarray(PS_DATA["s13"]))
PS_HIGH = np.maximum(np.asarray(PS_DATA["s12"]), np.asarray(PS_DATA["s13"]))
GEN_INTENSITY = {q: np.asarray(GEN_MODELS[q].intensity(PS_DATA)) for q in CHARGES}


def isobar_s_wave_1d(masses):
    # S_iso(m): sum over sigma and rescattering of c_j N_j A_j(m) (spin 0: lineshape only).
    model = ISOBAR_MODELS[+1]
    masses = jnp.asarray(masses, dtype=float)
    total = 0.0
    for component in model.amplitude_model.components:
        if component.name not in S_WAVE_ISOBARS:
            continue
        coefficient = complex(np.asarray(component.coefficient.value({})))
        scale = float(np.asarray(model._component_scale(component, {})))
        context = component.function.context.resolve({})
        total = total + coefficient * scale * np.asarray(component.function.lineshape(masses, context))
    return total


def edges_centred_on_points(points):
    # Bin edges at the midpoints (in s) between points; end bins mirrored around the end points.
    centres = np.sort(np.asarray(points)) ** 2
    midpoints = 0.5 * (centres[:-1] + centres[1:])
    edges = np.r_[2 * centres[0] - midpoints[0], midpoints, 2 * centres[-1] - midpoints[-1]]
    return tuple(float(v) for v in edges)


# Fine 1D description of the S-wave, weighted by the phase-space density of m_low (the same
# weighting as the 2D grid mismodelling; weighting by the full intensity puts all points on the
# rho(770) peak).
M_FINE = np.linspace(2 * M_PI, M_B - M_PI, 20_000)
S_FINE = isobar_s_wave_1d(M_FINE)
PHASE_FINE = np.unwrap(np.angle(S_FINE))
_m_bins = np.linspace(2 * M_PI, M_B - M_PI, 2_001)
_density, _ = np.histogram(np.sqrt(PS_LOW), bins=_m_bins)
WEIGHT_FINE = np.interp(M_FINE, 0.5 * (_m_bins[:-1] + _m_bins[1:]), _density)


def interpolation_error_1d(points):
    # Event-weighted error of the linear (in s) magnitude/phase interpolation through the centres.
    edges = np.asarray(edges_centred_on_points(points))
    centres = 0.5 * (edges[:-1] + edges[1:])
    centre_masses = np.sqrt(centres)
    magnitude = np.interp(M_FINE**2, centres, np.interp(centre_masses, M_FINE, np.abs(S_FINE)))
    phase = np.interp(M_FINE**2, centres, np.interp(centre_masses, M_FINE, PHASE_FINE))
    residual = np.abs(magnitude * np.exp(1j * phase) - S_FINE) ** 2
    return float(np.sum(WEIGHT_FINE * residual) / np.sum(WEIGHT_FINE * np.abs(S_FINE) ** 2))


ADAPTIVE_POINTS = [2 * M_PI, M_B - M_PI]
_candidates = np.arange(0.30, M_B - M_PI - MIN_POINT_SPACING, 0.005)
adaptive_history = [(None, interpolation_error_1d(ADAPTIVE_POINTS))]
while len(ADAPTIVE_POINTS) < N_POINTS_1D:
    trials = {
        float(m): interpolation_error_1d(ADAPTIVE_POINTS + [float(m)])
        for m in _candidates
        if min(abs(m - p) for p in ADAPTIVE_POINTS) >= MIN_POINT_SPACING
    }
    best = min(trials, key=trials.get)
    ADAPTIVE_POINTS.append(best)
    adaptive_history.append((best, trials[best]))
ADAPTIVE_POINTS = tuple(sorted(ADAPTIVE_POINTS))
print("adaptive 1D points m [GeV]:", np.round(ADAPTIVE_POINTS, 3))
print(pd.DataFrame(adaptive_history, columns=["added m [GeV]", "1D relative error"]).to_string(index=False))

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, constrained_layout=True)
shown = M_FINE <= 3.2
axes[0].plot(M_FINE[shown], np.abs(S_FINE[shown]), color="#4C72B0")
axes[0].set_ylabel(r"$|S_\mathrm{iso}|$")
axes[1].plot(M_FINE[shown], PHASE_FINE[shown], color="#4C72B0")
axes[1].set_ylabel(r"arg $S_\mathrm{iso}$ [rad]")
mass_bins = np.arange(2 * M_PI, 3.2 + 0.02, 0.02)
low_yield, _ = np.histogram(np.sqrt(PS_LOW), bins=mass_bins,
                            weights=N_TOY_EVENTS * GEN_INTENSITY[+1] / GEN_INTENSITY[+1].sum())
axes[2].stairs(low_yield, mass_bins, color="black")
axes[2].set(ylabel="expected events / 20 MeV", xlabel=r"$m(\pi^+\pi^-)_\mathrm{low}$ [GeV]")
_centre_masses = np.sqrt(0.5 * (np.asarray(edges_centred_on_points(ADAPTIVE_POINTS))[:-1]
                                + np.asarray(edges_centred_on_points(ADAPTIVE_POINTS))[1:]))
for ax in axes:
    for centre in _centre_masses[_centre_masses <= 3.2]:
        ax.axvline(centre, color="#C44E52", lw=0.8, zorder=0)
axes[0].set_title("generated S-wave and the adaptive 1D binning (red: cell centres)")
plt.show()

In [ ]:
S_EDGES = edges_centred_on_points(ADAPTIVE_POINTS)
N_BINS = len(S_EDGES) - 1
S_CENTERS = 0.5 * (np.asarray(S_EDGES[:-1]) + np.asarray(S_EDGES[1:]))


def inside_dalitz(s12, s13):
    # Is the point (s12, s13) inside the Dalitz plot of B -> pi pi pi (all daughters m_pi)?
    if not (S_MIN < s12 < S_MAX):
        return False
    m = M_PI
    e2 = 0.5 * np.sqrt(s12)  # energy of daughter 1 (or 2) in the 12 rest frame
    e3 = (M_B**2 - s12 - m**2) / (2 * np.sqrt(s12))
    p2, p3 = np.sqrt(max(e2**2 - m**2, 0.0)), np.sqrt(max(e3**2 - m**2, 0.0))
    return (e2 + e3) ** 2 - (p2 + p3) ** 2 < s13 < (e2 + e3) ** 2 - (p2 - p3) ** 2


ACTIVE = np.array([[inside_dalitz(S_CENTERS[i], S_CENTERS[j]) for j in range(N_BINS)]
                   for i in range(N_BINS)])
assert np.array_equal(ACTIVE, ACTIVE.T)
# One parameter set per mirrored pair: the canonical cell (i, j) with i <= j carries it.
ACTIVE_CELLS = [(i, j) for i, j in (tuple(int(v) for v in c) for c in np.argwhere(ACTIVE)) if i <= j]
physical = np.zeros_like(ACTIVE)  # canonical active cells, for the folded maps and tables
for cell in ACTIVE_CELLS:
    physical[cell] = True

# Expected events per canonical cell (folded coordinates) from the phase-space sample.
_ix = np.clip(np.searchsorted(S_EDGES, PS_LOW, side="right") - 1, 0, N_BINS - 1)
_iy = np.clip(np.searchsorted(S_EDGES, PS_HIGH, side="right") - 1, 0, N_BINS - 1)
expected_events = {}
for charge in CHARGES:
    counts = np.zeros((N_BINS, N_BINS))
    np.add.at(counts, (_ix, _iy), N_TOY_EVENTS * GEN_INTENSITY[charge] / GEN_INTENSITY[charge].sum())
    expected_events[charge] = counts
EXPECTED_MIN = np.minimum.reduce([expected_events[q] for q in CHARGES])

print(f"{N_BINS} x {N_BINS} cells on the unfolded plane, {int(ACTIVE.sum())} centres inside the "
      f"Dalitz plot, {len(ACTIVE_CELLS)} independent (mirrored) cells")
print("fewest expected events in an active cell:", round(float(EXPECTED_MIN[physical].min()), 1))

### Reference values at the cell centres

$S_\text{gen}$ is evaluated at every active cell centre. Phases are made continuous by a
breadth-first walk over neighbouring active cells of the unfolded plane (each new phase is the
neighbour's phase plus the wrapped difference), started at the most populated cell; this requires
the active cells to be connected and $S_\text{gen}$ to have no zero, both checked. $S_\text{gen}$ is
symmetric, so mirrored cells get the same reference values.

In [ ]:
def center_data(cells):
    s12 = jnp.asarray([S_CENTERS[i] for i, _ in cells])
    s13 = jnp.asarray([S_CENTERS[j] for _, j in cells])
    return {"s12": s12, "s13": s13, "s23": S_SUM - s12 - s13}


def continuous_phases(values, cells, start):
    index = {cell: k for k, cell in enumerate(cells)}
    raw = np.angle(values)
    phases = np.full(len(cells), np.nan)
    phases[index[start]] = raw[index[start]]
    queue = deque([start])
    while queue:
        i, j = queue.popleft()
        for neighbour in ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1)):
            k = index.get(neighbour)
            if k is None or np.isfinite(phases[k]):
                continue
            here = phases[index[(i, j)]]
            phases[k] = here + np.angle(np.exp(1j * (raw[k] - here)))
            queue.append(neighbour)
    assert np.all(np.isfinite(phases)), "active cells are not connected"
    return phases


ALL_ACTIVE_CELLS = [tuple(int(v) for v in cell) for cell in np.argwhere(ACTIVE)]
_start = max(ACTIVE_CELLS, key=lambda cell: EXPECTED_MIN[cell])
_canonical_index = [ALL_ACTIVE_CELLS.index(cell) for cell in ACTIVE_CELLS]
REFERENCE = {}  # charge -> (magnitudes, phases) on ACTIVE_CELLS
for charge in CHARGES:
    values = np.asarray(generated_s_wave(charge)(center_data(ALL_ACTIVE_CELLS)))
    assert np.all(np.isfinite(values))
    phases = continuous_phases(values, ALL_ACTIVE_CELLS, _start)
    mirror = [ALL_ACTIVE_CELLS.index((j, i)) for i, j in ACTIVE_CELLS]
    assert np.allclose(phases[_canonical_index], phases[mirror])
    REFERENCE[charge] = (np.abs(values)[_canonical_index], phases[_canonical_index])

_s_gen_ps = {q: np.asarray(generated_s_wave(q)(PS_DATA)) for q in CHARGES}
for charge in CHARGES:
    ratio = np.min(np.abs(_s_gen_ps[charge])) / np.sqrt(np.mean(np.abs(_s_gen_ps[charge]) ** 2))
    print(f"charge {charge:+d}: min |S_gen| / rms |S_gen| over phase space = {ratio:.3f}")
    assert ratio > 0.01

## Fit model: P/D/F isobars + QMIPixel S-wave

In [ ]:
TRUE_VALUES = {}


def coefficient_parameters(name, values):
    pars = tuple(
        Parameter.coefficient(f"{name}.{axis}", float(value), owner=name, step=0.01,
                              fixed=f"{name}.{axis}" in FIXED_COEFFICIENTS)
        for axis, value in zip(("x", "y"), values[:2], strict=True)
    )
    TRUE_VALUES.update({p.name: p.value for p in pars})
    return RealImag(*pars)


shared_coefficients = {
    name: coefficient_parameters(name, values)
    for name, values in COEFFICIENTS.items() if name not in S_WAVE_ISOBARS
}


def cell_name(cell, quantity, charge=+1):
    # Without CP violation there is a single S-wave; `charge` is kept for the shared helpers.
    return f"S_QMIPixel[{cell[0]},{cell[1]}].{quantity}"


def qmi_pixel_parameters(charge):
    # One magnitude and one phase per active cell; QMIPixel nodes live inside the dynamics, so they
    # must be Parameter.dynamics owned by the S_QMIPixel component.
    magnitudes, phases = REFERENCE[charge]
    grid_mag = [[0.0] * N_BINS for _ in range(N_BINS)]
    grid_phase = [[0.0] * N_BINS for _ in range(N_BINS)]
    for cell, magnitude, phase in zip(ACTIVE_CELLS, magnitudes, phases, strict=True):
        i, j = cell
        # The mirrored cell (j, i) shares the same Parameter objects: symmetric field.
        grid_mag[i][j] = grid_mag[j][i] = Parameter.dynamics(
            cell_name(cell, "mag", charge), float(magnitude), owner="S_QMIPixel",
            step=max(0.1 * float(magnitude), 1e-3),
        )
        grid_phase[i][j] = grid_phase[j][i] = Parameter.dynamics(
            cell_name(cell, "phase", charge), float(phase), owner="S_QMIPixel", step=0.05,
        )
        TRUE_VALUES[grid_mag[i][j].name] = float(magnitude)
        TRUE_VALUES[grid_phase[i][j].name] = float(phase)
    return tuple(map(tuple, grid_mag)), tuple(map(tuple, grid_phase))


def s_wave_qmi_pixel(charge, magnitudes=None, phases=None):
    if magnitudes is None:
        magnitudes, phases = qmi_pixel_parameters(charge)
    field = QMIPixel(
        s12_edges=S_EDGES, s13_edges=S_EDGES, magnitudes=magnitudes, phases=phases,
        interpolation=QMI_PIXEL_INTERPOLATION, folded=False,
        active_mask=tuple(map(tuple, ACTIVE.tolist())),
    )
    # Fixed 1+0i coefficient and no unit normalization: the cells carry the physical S-wave.
    return DalitzAmplitude("S_QMIPixel", field, RealImag(1.0, 0.0), normalize_component=False)


def build_components(charge):
    components = [
        isobar_resonance(name, coefficient)
        for name, coefficient in shared_coefficients.items()
    ]
    return components + [s_wave_qmi_pixel(charge)]


FIT_MODEL = build_model(+1, build_components(+1))
FREE_PARAMETER_NAMES = list(dict.fromkeys(p.name for p in FIT_MODEL.parameters if not p.fixed))
PDF_WAVE_PARAMETER_NAMES = [n for n in FREE_PARAMETER_NAMES if not n.startswith("S_QMIPixel")]
CELL_PARAMETER_NAMES = [n for n in FREE_PARAMETER_NAMES if n.startswith("S_QMIPixel")]
assert set(FREE_PARAMETER_NAMES) == set(TRUE_VALUES) - FIXED_COEFFICIENTS
print("fit components:", [c.name for c in FIT_MODEL.amplitude_model.components])
print(f"free parameters: {len(FREE_PARAMETER_NAMES)} "
      f"({len(PDF_WAVE_PARAMETER_NAMES)} P/D/F, {len(CELL_PARAMETER_NAMES)} QMIPixel)")

## Grid mismodelling: QMIPixel at the reference values vs. generated S-wave

The QMIPixel evaluated at the reference values is the fit model at its starting point. Its distance
to $S_\text{gen}$ bounds what the grid can represent; the fit can absorb part of it by moving the
cells away from the reference values. Two figures of merit over phase space:
relative complex RMS of the S-wave, and $\int |S_\text{QMIPixel} - S_\text{gen}|^2 / \int |A_\text{gen}|^2$
(the mismodelled intensity relative to the full signal).

In [ ]:
def numeric_field(charge, values):
    # QMIPixel with numeric nodes from a {parameter name: value} mapping.
    magnitudes = [[0.0] * N_BINS for _ in range(N_BINS)]
    phases = [[0.0] * N_BINS for _ in range(N_BINS)]
    for i, j in ACTIVE_CELLS:
        magnitudes[i][j] = magnitudes[j][i] = float(values[cell_name((i, j), "mag", charge)])
        phases[i][j] = phases[j][i] = float(values[cell_name((i, j), "phase", charge)])
    return s_wave_qmi_pixel(charge, tuple(map(tuple, magnitudes)), tuple(map(tuple, phases))).dynamics


fig, axes = plt.subplots(len(CHARGES), 3, figsize=(16, 4.5 * len(CHARGES)), constrained_layout=True,
                         squeeze=False)
for row, charge in enumerate(CHARGES):
    s_gen = _s_gen_ps[charge]
    s_qmi = np.asarray(numeric_field(charge, TRUE_VALUES)(PS_DATA))
    amplitude = np.asarray(GEN_MODELS[charge].amplitude(PS_DATA))
    relative_rms = np.sqrt(np.mean(np.abs(s_qmi - s_gen) ** 2) / np.mean(np.abs(s_gen) ** 2))
    intensity_share = np.sum(np.abs(s_qmi - s_gen) ** 2) / np.sum(np.abs(amplitude) ** 2)
    label = "B+" if charge > 0 else "B-"
    print(f"{label}: relative complex RMS = {relative_rms:.3f}, "
          f"int|dS|^2 / int|A|^2 = {intensity_share:.2e}")

    panels = (
        (np.abs(s_gen), r"$|S_\mathrm{gen}|$", "viridis", None),
        (np.angle(s_gen), r"arg $S_\mathrm{gen}$ [rad]", "twilight", (-np.pi, np.pi)),
        (np.abs(s_qmi - s_gen) / np.sqrt(np.mean(np.abs(s_gen) ** 2)),
         r"$|S_\mathrm{QMIPixel} - S_\mathrm{gen}|$ / rms$|S_\mathrm{gen}|$", "magma", (0, 0.5)),
    )
    for ax, (values, title, cmap, limits) in zip(axes[row], panels):
        sc = ax.scatter(PS_LOW[::5], PS_HIGH[::5], c=values[::5], s=1, cmap=cmap,
                        vmin=None if limits is None else limits[0],
                        vmax=None if limits is None else limits[1], rasterized=True)
        for edge in S_EDGES:
            ax.axvline(edge, color="white", lw=0.3)
            ax.axhline(edge, color="white", lw=0.3)
        ax.set(xscale="log", xlabel=r"$s_\mathrm{low}$ [GeV$^2$]", ylabel=r"$s_\mathrm{high}$ [GeV$^2$]",
               title=f"{label}: {title}")
        fig.colorbar(sc, ax=ax)
plt.show()


def unexplained_fractions(sample=200_000):
    # Fraction of each P/D wave's int|A_k|^2 that the cells cannot reproduce (linearized least
    # squares over phase space, real hat functions of the mirrored cells as basis).
    data = {key: value[:sample] for key, value in PS_DATA.items()}
    basis = []
    for i, j in ACTIVE_CELLS:
        unit = [[0.0] * N_BINS for _ in range(N_BINS)]
        unit[i][j] = unit[j][i] = 1.0
        zeros = tuple((0.0,) * N_BINS for _ in range(N_BINS))
        field = s_wave_qmi_pixel(+1, tuple(map(tuple, unit)), zeros).dynamics
        basis.append(np.asarray(field.interpolated_magnitude_phase(data)[0]))
    basis = np.array(basis).T
    components = GEN_MODELS[+1].component_amplitudes(data)
    result = {}
    for name in ("rho770", "f2_1270", "rho1450", "rho3_1690"):
        target = np.asarray(components[name])
        solution, *_ = np.linalg.lstsq(basis, target, rcond=None)
        result[name] = float(np.sum(np.abs(target - basis @ solution) ** 2) / np.sum(np.abs(target) ** 2))
    return result


print("fraction of each P/D wave the QMIPixel cells cannot reproduce:",
      {name: round(value, 3) for name, value in unexplained_fractions().items()})

### Phase along $s_\text{high}$

Reference phases of a few $s_\text{low}$ columns vs. the cell-centre $s_\text{high}$ (the isobaric
S-wave phase varies mostly with $s_\text{low}$; the $s_\text{high}$ dependence comes from the
second pair, $S_\text{iso}(s_\text{high})$).

In [ ]:
def column_cells(i):
    return [cell for cell in ACTIVE_CELLS if cell[0] == i]


SHOWN_COLUMNS = [i for i in range(N_BINS) if len(column_cells(i)) >= 4][:6]
fig, axes = plt.subplots(1, len(CHARGES), figsize=(6.5 * len(CHARGES), 4), constrained_layout=True,
                         squeeze=False)
for ax, charge in zip(axes[0], CHARGES):
    _, phases = REFERENCE[charge]
    for i in SHOWN_COLUMNS:
        cells = column_cells(i)
        k = [ACTIVE_CELLS.index(cell) for cell in cells]
        ax.plot([S_CENTERS[j] for _, j in cells], phases[k], "o-", ms=3,
                label=fr"$s_\mathrm{{low}}$ bin {i} ({S_EDGES[i]:.2f}-{S_EDGES[i + 1]:.2f})")
    ax.set(xlabel=r"$s_\mathrm{high}$ cell centre [GeV$^2$]", ylabel="reference phase [rad]",
           title="B+" if charge > 0 else "B-")
axes[0, 0].legend(fontsize=7)
plt.show()

## Projections before the fit: isobar model vs. QMIPixel at the reference values

Before any toy is fitted: the generation (isobar) model against the QMIPixel fit model at its
starting point, the reference values (P/D/F coefficients of Table XXI and the generated S-wave
at the cell centres). This is what the pixel grid can represent with the exact P/D/F waves; the
fit can only improve on it. All curves are projections of the normalized intensity of the same
uniform phase-space sample (`PS_DATA`, 1M events) scaled to `N_TOY_EVENTS`, so the comparison has
no toy statistical fluctuation. Lower panels: per-bin pulls
$(N_\text{QMIPixel} - N_\text{iso}) / \sqrt{N_\text{iso}}$, i.e. the difference in units of the
Poisson uncertainty of one toy (band: $\pm 2$). The second figure shows the S-wave alone
($|S|^2$, no interference), scaled the same way; vertical lines: the cell centres.

In [ ]:
PROJECTION_STYLES = {
    "isobar (generation)": dict(color="black", lw=1.4),
    "QMIPixel, toy-mean parameters": dict(color="#C44E52", lw=1.2),
    "QMIPixel, reference values": dict(color="#4C72B0", lw=1.0, ls="--"),
}
PROJECTION_VARIABLES = (
    (np.sqrt(PS_LOW), np.linspace(2 * M_PI, 2.0, 90), r"$m(\pi^+\pi^-)_\mathrm{low}$ [GeV], zoom"),
    (np.sqrt(PS_LOW), np.linspace(2 * M_PI, np.sqrt(S_SUM / 2), 90), r"$m(\pi^+\pi^-)_\mathrm{low}$ [GeV]"),
    (np.sqrt(PS_HIGH), np.linspace(1.0, M_B - M_PI, 90), r"$m(\pi^+\pi^-)_\mathrm{high}$ [GeV]"),
    (np.sqrt(np.asarray(PS_DATA["s23"])), np.linspace(2 * M_PI, M_B - M_PI, 90),
     r"$m(\pi^+\pi^+)$ [GeV]"),
)


def poisson_pulls(yields, reference):
    # (N_QMIPixel - N_iso) / sqrt(N_iso), zero where the isobar model predicts no events.
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(reference > 0, (yields - reference) / np.sqrt(reference), 0.0)


def projected_yields(weights, variable, edges, total):
    counts, _ = np.histogram(variable, bins=edges, weights=weights)
    return N_TOY_EVENTS * counts / total


def _projection_figure(curves, variables, total, ylabel, pull_limit, centre_lines=False):
    # curves: {label: per-event weights}; the isobar curve is the reference of the pulls.
    fig, axes = plt.subplots(2, len(variables), figsize=(5 * len(variables), 7), sharex="col",
                             constrained_layout=True, gridspec_kw={"height_ratios": [3, 1.3]},
                             squeeze=False)
    for column, (variable, edges, label) in enumerate(variables):
        reference = projected_yields(curves["isobar (generation)"], variable, edges, total)
        for name, weights in curves.items():
            yields = projected_yields(weights, variable, edges, total)
            axes[0, column].stairs(yields, edges, label=name, **PROJECTION_STYLES[name])
            if name != "isobar (generation)":
                axes[1, column].stairs(poisson_pulls(yields, reference), edges, **PROJECTION_STYLES[name])
                if column == 1:
                    mask = reference > 0
                    chi2 = np.sum((yields[mask] - reference[mask]) ** 2 / reference[mask])
                    print(f"{name}: sum over {label} bins of (QMIPixel - isobar)^2 / isobar = "
                          f"{chi2:.1f} for {mask.sum()} bins (one-toy Poisson scale)")
        if centre_lines:
            for centre in np.sqrt(S_CENTERS):
                if centre <= edges[-1]:
                    for ax in axes[:, column]:
                        ax.axvline(centre, color="0.7", lw=0.5, zorder=0)
        axes[1, column].axhspan(-2, 2, color="0.9", zorder=0)
        axes[1, column].axhline(0.0, color="black", lw=0.8)
        axes[1, column].set_ylim(-pull_limit, pull_limit)
        axes[1, column].set_xlabel(label)
        axes[0, column].set_ylabel(ylabel)
    axes[1, 0].set_ylabel("pull")
    axes[0, 0].legend(fontsize=8)
    plt.show()


def plot_projections(qmi_pixel_intensities):
    # Full intensity: each curve normalized to N_TOY_EVENTS on its own.
    curves = {"isobar (generation)": GEN_INTENSITY[+1], **qmi_pixel_intensities}
    fig_curves = {name: weights / weights.sum() for name, weights in curves.items()}
    _projection_figure(fig_curves, PROJECTION_VARIABLES, 1.0,
                       f"events / bin ({N_TOY_EVENTS} per toy)", 5)


def plot_s_wave_projections(qmi_pixel_s_waves):
    # S-wave alone, |S|^2 on the scale of the full projections (S-wave events if it were alone).
    curves = {"isobar (generation)": np.abs(_s_gen_ps[+1]) ** 2, **qmi_pixel_s_waves}
    _projection_figure(curves, PROJECTION_VARIABLES[:2], GEN_INTENSITY[+1].sum(),
                       r"$|S|^2$ events / bin", 8, centre_lines=True)


plot_projections({"QMIPixel, reference values": np.asarray(FIT_MODEL.intensity(PS_DATA, TRUE_VALUES))})
plot_s_wave_projections({
    "QMIPixel, reference values": np.abs(np.asarray(numeric_field(+1, TRUE_VALUES)(PS_DATA))) ** 2,
})

## Gen-fit loop

For each `iExpt`: generate one toy from the isobar model (accept-reject, deterministic seed), fit
it with the QMIPixel model through `FitSession` starting from the reference values, and append one
row to `RESULTS_CSV` with the fit status and the value and HESSE error of every free parameter
(the QMIPixel parameters are already the magnitude and phase of every cell). Toys already
in `RESULTS_CSV` are skipped; a toy whose generation or fit raises is reported and skipped (it is
retried on the next run).

In [ ]:
import gc

import jax


def release_toy_memory():
    # Every toy builds new jitted closures; free their cached programs (see the 1D notebook).
    gc.collect()
    jax.clear_caches()
    gc.collect()


COMPILATION_CACHE_DIR = Path(os.environ["JAX_COMPILATION_CACHE_DIR"])
generator_programs_cached = False  # set once this kernel has kept the generator programs


def compilation_cache_entries():
    return set(COMPILATION_CACHE_DIR.iterdir()) if COMPILATION_CACHE_DIR.is_dir() else set()


def discard_new_cache_entries(kept):
    # Fit programs embed the toy's events and can never be reused: drop what this toy wrote.
    for entry in compilation_cache_entries() - kept:
        entry.unlink(missing_ok=True)


def toy_seed(iexpt):
    return int(np.random.SeedSequence([TOY_BASE_SEED, iexpt]).generate_state(1)[0])


def make_toy(iexpt):
    return generate_toy(
        GEN_MODELS[+1], N_TOY_EVENTS,
        parameters={},
        method="accept-reject",
        pool_size=TOY_POOL_SIZE,
        include_momenta=False,
        seed=toy_seed(iexpt),
    )


def run_toy(iexpt):
    global generator_programs_cached
    cache_snapshot = compilation_cache_entries()
    try:
        t0 = time.perf_counter()
        data = make_toy(iexpt)
        gen_time = time.perf_counter() - t0
        if not generator_programs_cached:
            cache_snapshot = compilation_cache_entries()
            generator_programs_cached = True

        session = FitSession(FIT_MODEL, data)
        t0 = time.perf_counter()
        result = session.fit(
            dict(TRUE_VALUES), strategy=FIT_STRATEGY, ncall=FIT_NCALL,
            hesse=True, hessian="jax", verbose=0,
        )
        fit_time = time.perf_counter() - t0
    finally:
        discard_new_cache_entries(cache_snapshot)
    values = session.result_values(result)

    row = {
        "iExpt": iexpt,
        "seed": toy_seed(iexpt),
        "n_events": data.size,
        "valid": bool(result.valid),
        "accurate_covariance": bool(result.fmin.has_accurate_covar),
        "nll": float(result.fval),
        "edm": float(result.fmin.edm),
        "nfcn": int(result.nfcn),
        "gen_time_s": gen_time,
        "fit_time_s": fit_time,
    }
    for name in FREE_PARAMETER_NAMES:
        row[name] = float(values[name])
        row[f"{name}_error"] = float(result.errors[name])
    return row


completed = (
    set(pd.read_csv(RESULTS_CSV)["iExpt"].astype(int)) if RESULTS_CSV.exists() else set()
)
pending = [i for i in TOY_INDICES if i not in completed]
print(f"{len(completed)} toys already fitted, {len(pending)} pending")

for iexpt in pending:
    try:
        row = run_toy(iexpt)
    except Exception as exc:  # keep the loop alive across a single bad toy
        print(f"iExpt={iexpt}: FAILED ({exc!r})")
        continue
    finally:
        release_toy_memory()
    pd.DataFrame([row]).to_csv(
        RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False
    )
    print(f"iExpt={iexpt}: valid={row['valid']} nll={row['nll']:.2f} "
          f"events={row['n_events']} nfcn={row['nfcn']} "
          f"(gen {row['gen_time_s']:.1f}s, fit {row['fit_time_s']:.1f}s)", flush=True)

## Results

Only toys with `valid=True` enter the statistics. Phase residuals are wrapped to $(-\pi, \pi]$
before computing pulls. For the QMIPixel cells the "true value" is the reference value at the cell
centre, so their pulls also measure the grid approximation.

In [ ]:
toy_results = pd.read_csv(RESULTS_CSV)
n_valid = int(toy_results["valid"].sum())
print(f"{len(toy_results)} toys fitted, {n_valid} with valid=True, "
      f"{int(toy_results['accurate_covariance'].sum())} with accurate covariance")
print(f"fit time: median {toy_results['fit_time_s'].median():.0f}s, "
      f"nfcn median {toy_results['nfcn'].median():.0f}")

toy_results_valid = toy_results[toy_results["valid"]].copy()


def wrap_phase(delta):
    return np.angle(np.exp(1j * np.asarray(delta, dtype=float)))


derived_columns = {}
for name in FREE_PARAMETER_NAMES:
    residual = toy_results_valid[name] - TRUE_VALUES[name]
    if name.endswith(".phase"):
        residual = wrap_phase(residual)
    derived_columns[f"{name}_residual"] = residual
    derived_columns[f"{name}_pull"] = residual / toy_results_valid[f"{name}_error"]
toy_results_valid = pd.concat(
    [toy_results_valid, pd.DataFrame(derived_columns, index=toy_results_valid.index)], axis=1
)


def ensemble_stats(name):
    residuals = toy_results_valid[f"{name}_residual"].to_numpy()
    pulls = toy_results_valid[f"{name}_pull"].to_numpy()
    n = np.isfinite(pulls).sum()
    return {
        "parameter": name, "true_value": TRUE_VALUES[name],
        "value_mu": TRUE_VALUES[name] + np.nanmean(residuals),
        "value_sigma": np.nanstd(residuals, ddof=1),
        "error_mu": np.nanmean(toy_results_valid[f"{name}_error"]),
        "pull_mu": np.nanmean(pulls), "pull_mu_error": np.nanstd(pulls, ddof=1) / np.sqrt(n),
        "pull_sigma": np.nanstd(pulls, ddof=1),
        "pull_sigma_error": np.nanstd(pulls, ddof=1) / np.sqrt(2.0 * (n - 1)),
    }


pull_summary = pd.DataFrame([ensemble_stats(n) for n in FREE_PARAMETER_NAMES]).set_index("parameter")
pull_summary["bias_over_spread"] = (
    (pull_summary["value_mu"] - pull_summary["true_value"]) / pull_summary["value_sigma"]
)
pull_summary.to_csv(RESULTS_CSV.with_name(RESULTS_CSV.stem + "_pull_summary.csv"))

## P/D/F-wave parameters

The P/D/F-wave coefficients have exact true values (Table XXI) in both models, so any bias here is
induced by replacing the S-wave with the QMIPixel.

In [ ]:
def _gaussian(x, amplitude, mu, sigma):
    return amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


n_bins = int(np.clip(np.sqrt(len(toy_results_valid)), 5, 30))
fig, axes = plt.subplots(len(PDF_WAVE_PARAMETER_NAMES), 3,
                         figsize=(15.5, 3.0 * len(PDF_WAVE_PARAMETER_NAMES)), constrained_layout=True)
for row, name in enumerate(PDF_WAVE_PARAMETER_NAMES):
    stats = pull_summary.loc[name]
    columns = (
        (toy_results_valid[name], "fitted value", stats["true_value"]),
        (toy_results_valid[f"{name}_error"], "HESSE error", None),
        (toy_results_valid[f"{name}_pull"], "(fitted - true) / error", 0.0),
    )
    for ax, (data, xlabel, line) in zip(axes[row], columns):
        data = np.asarray(data, dtype=float)
        data = data[np.isfinite(data)]
        bins = np.linspace(-5, 5, n_bins + 1) if line == 0.0 else n_bins
        counts, edges, _ = ax.hist(data, bins=bins, color="#4C72B0", edgecolor="white")
        mu, sigma = data.mean(), data.std(ddof=1)
        x = np.linspace(edges[0], edges[-1], 300)
        width = edges[1] - edges[0]
        ax.plot(x, _gaussian(x, len(data) * width / (np.sqrt(2 * np.pi) * sigma), mu, sigma),
                color="#DD8452")
        if line is not None:
            ax.axvline(line, color="black", linestyle="--")
        ax.set_title(fr"{name}: $\mu$={mu:.4g}, $\sigma$={sigma:.3g}", fontsize=9)
        ax.set_xlabel(xlabel)
plt.show()

pdf_wave_summary = pull_summary.loc[PDF_WAVE_PARAMETER_NAMES, [
    "true_value", "value_mu", "value_sigma", "bias_over_spread", "error_mu",
    "pull_mu", "pull_mu_error", "pull_sigma", "pull_sigma_error",
]]
fig, ax = plt.subplots(figsize=(13, 4), constrained_layout=True)
positions = np.arange(len(pdf_wave_summary))
ax.axhspan(-0.2, 0.2, color="0.9")
ax.axhline(0.0, color="black", lw=0.8)
ax.errorbar(positions, pdf_wave_summary["pull_mu"], yerr=pdf_wave_summary["pull_sigma"],
            fmt="o", color="#4C72B0", capsize=3)
ax.set_xticks(positions, pdf_wave_summary.index, rotation=60, ha="right")
ax.set_ylabel("pull μ ± σ")
ax.set_title("P/D/F-wave coefficients: isobar generation, QMIPixel fit")
plt.show()
pdf_wave_summary

## S-wave recovery: maps of the cell pulls

Pull mean and pull width of every active cell, for the magnitude and the phase, on the
folded plane (axes in cell index; the cell edges are listed above). An unbiased cell with correct
errors has pull mean ≈ 0 and width ≈ 1. Cells whose magnitude is small have poorly defined phases.

In [ ]:
def cell_map(quantity, charge, column):
    grid = np.full((N_BINS, N_BINS), np.nan)
    for cell in ACTIVE_CELLS:
        grid[cell] = pull_summary.loc[cell_name(cell, quantity, charge), column]
    return grid


for column, limits, cmap in (("pull_mu", (-1, 1), "coolwarm"), ("pull_sigma", (0.5, 1.5), "PiYG")):
    fig, axes = plt.subplots(2, len(CHARGES), figsize=(7 * len(CHARGES), 11), constrained_layout=True,
                             squeeze=False)
    for r, quantity in enumerate(("mag", "phase")):
        for c, charge in enumerate(CHARGES):
            ax = axes[r, c]
            grid = cell_map(quantity, charge, column)
            image = ax.imshow(grid.T, origin="lower", cmap=cmap, vmin=limits[0], vmax=limits[1])
            for i, j in ACTIVE_CELLS:
                ax.text(i, j, f"{grid[i, j]:.1f}", ha="center", va="center", fontsize=5)
            ax.set(title=f"{quantity}: {column}",
                   xlabel=r"$s_\mathrm{low}$ bin", ylabel=r"$s_\mathrm{high}$ bin")
            fig.colorbar(image, ax=ax)
    plt.show()

cell_pulls = pull_summary.loc[CELL_PARAMETER_NAMES]
print("QMIPixel cells: pull mean of means = {:.3f}, mean pull width = {:.3f}".format(
    cell_pulls["pull_mu"].mean(), cell_pulls["pull_sigma"].mean()))
print("cells with |pull mean| > 0.5:")
print(cell_pulls[cell_pulls["pull_mu"].abs() > 0.5][["true_value", "value_mu", "pull_mu", "pull_sigma"]])

### Phase and magnitude along $s_\text{high}$, toy ensemble vs. reference

For the $s_\text{low}$ columns shown earlier: reference (line + black dots) and toy mean ± spread
(red). This checks that the fit follows the generated S-wave
along $s_\text{high}$.

In [ ]:
fig, axes = plt.subplots(2, len(SHOWN_COLUMNS), figsize=(3.2 * len(SHOWN_COLUMNS), 6.5),
                         constrained_layout=True, squeeze=False)
for c, i in enumerate(SHOWN_COLUMNS):
    cells = column_cells(i)
    s_high = np.array([S_CENTERS[j] for _, j in cells])
    for r, quantity in enumerate(("mag", "phase")):
        ax = axes[r, c]
        for charge, color in zip(CHARGES, ("#C44E52", "#4C72B0")):
            names = [cell_name(cell, quantity, charge) for cell in cells]
            truth = pull_summary.loc[names, "true_value"].to_numpy()
            mean = pull_summary.loc[names, "value_mu"].to_numpy()
            spread = pull_summary.loc[names, "value_sigma"].to_numpy()
            ax.plot(s_high, truth, "-", color=color, lw=0.8)
            ax.plot(s_high, truth, "o", color="black", ms=3)
            ax.errorbar(s_high, mean, yerr=spread, fmt="s", ms=3, color=color, capsize=2,
                        label="QMIPixel fit: toy mean ± spread")
        ax.set_title(fr"$s_\mathrm{{low}}\in$[{S_EDGES[i]:.2f}, {S_EDGES[i + 1]:.2f}]", fontsize=9)
        ax.set_xlabel(r"$s_\mathrm{high}$ [GeV$^2$]")
        ax.set_ylabel("magnitude" if quantity == "mag" else "phase [rad]")
axes[0, 0].legend(fontsize=8)
plt.show()

### QMIPixel cell table

Per cell and quantity: reference, toy mean, bias in units of the toy spread, toy spread
vs. mean HESSE error, and pull mean/width. Saved next to `RESULTS_CSV`.

In [ ]:
cell_rows = []
for charge in CHARGES:
    for quantity in ("mag", "phase"):
        for cell in ACTIVE_CELLS:
            stats = pull_summary.loc[cell_name(cell, quantity, charge)]
            cell_rows.append({
                "charge": "B+" if charge > 0 else "B-", "quantity": quantity,
                "cell": f"{cell[0]},{cell[1]}",
                "s_low": S_CENTERS[cell[0]], "s_high": S_CENTERS[cell[1]],
                "expected_events": expected_events[charge][cell],
                "reference": stats["true_value"], "toy_mean": stats["value_mu"],
                "bias_over_spread": stats["bias_over_spread"],
                "toy_spread": stats["value_sigma"], "mean_hesse_error": stats["error_mu"],
                "pull_mu": stats["pull_mu"], "pull_sigma": stats["pull_sigma"],
            })
cell_summary = pd.DataFrame(cell_rows).set_index(["charge", "quantity", "cell"])
cell_summary.to_csv(RESULTS_CSV.with_name(RESULTS_CSV.stem + "_cell_summary.csv"))
cell_summary

## Projections: isobar model vs. QMIPixel at the toy-mean parameters

As in "Projections before the fit", now with the QMIPixel fit model at the **mean of the fitted
parameters over the valid toys** (`value_mu` of `pull_summary`; phases averaged through their
wrapped residuals); dashed: the reference values the fits start from.

In [ ]:
MEAN_VALUES = dict(TRUE_VALUES)
MEAN_VALUES.update(pull_summary["value_mu"].to_dict())
plot_projections({
    "QMIPixel, toy-mean parameters": np.asarray(FIT_MODEL.intensity(PS_DATA, MEAN_VALUES)),
    "QMIPixel, reference values": np.asarray(FIT_MODEL.intensity(PS_DATA, TRUE_VALUES)),
})
plot_s_wave_projections({
    "QMIPixel, toy-mean parameters": np.abs(np.asarray(numeric_field(+1, MEAN_VALUES)(PS_DATA))) ** 2,
    "QMIPixel, reference values": np.abs(np.asarray(numeric_field(+1, TRUE_VALUES)(PS_DATA))) ** 2,
})